# PaperRAG 公开论文检索测评

这个 notebook 只测 **检索 / 召回阶段**，不会调用 LLM 生成答案。

当前评测套件使用公开学术数据集：

- `Qasper`：测论文问答场景下的 paper recall 和 evidence recall
- `SciFact`：测科学文献相关性检索

输出结果包括 `summary.json`、`detail.csv` 和 `failures.csv`，便于做参数 A/B 对比。


## 使用方式

1. 先在下面的配置 cell 里填写 Milvus / API Key。
2. 如果你只想跑单个数据集，把 `TARGET` 改成 `qasper` 或 `scifact`。
3. `EXTRA_ENV_OVERRIDES` 可以临时覆盖 `.env.example` 里的检索参数，用于 A/B 对比。
4. notebook 默认从仓库里的 `.env.example` 读取非敏感参数，只在 Colab 里覆盖必填的密钥和运行时变量。


In [ ]:
from pathlib import Path

# ===== repo =====
REPO_OWNER = 'YaoHui-Wu06022'
REPO_NAME = 'PaperRAG'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/PaperRAG')
ENV_TEMPLATE_PATH = '.env.example'

# ===== benchmark target =====
# target: 'suite' / 'qasper' / 'scifact'
TARGET = 'suite'
LIMIT = 50
SEED = 88
SHUFFLE = True

# ===== output =====
OUTPUT_DIR = Path('/content/public_evalout')

# ===== vector db =====
# MILVUS_MODE: 'cloud' or 'lite'
MILVUS_MODE = 'lite'
MILVUS_URI = ''
MILVUS_TOKEN = ''
MILVUS_DB_NAME = ''
MILVUS_LITE_URI = '/content/milvus_lite/public_paper_eval.db'

# ===== API keys =====
# 如果 embedding 和 reranker 都用 Jina，可以直接只填 JINA_API_KEY。
JINA_API_KEY = ''
EMBEDDING_API_KEY = JINA_API_KEY
RERANKER_API_KEY = JINA_API_KEY

# ===== optional overrides =====
# 只在这里改你想对比的检索参数，例如：
# EXTRA_ENV_OVERRIDES = {'RETRIEVAL_MODE': 'dense', 'USE_RERANKER': 'false'}
EXTRA_ENV_OVERRIDES = {}


In [ ]:
import os
import shutil
import subprocess

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, clone_url, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('repo_ready=', REPO_DIR)


In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'python-dotenv>=1.0.1',
    'pymilvus>=2.5.0',
    'milvus-lite>=2.4.0',
    'langchain-milvus>=0.1.7',
    'langchain>=0.3.7',
    'langchain-core>=0.3.15',
    'langchain-community>=0.3.7',
    'langchain-text-splitters>=0.3.2',
    'langchain-huggingface>=0.1.0',
    'langchain-openai>=0.2.5',
    'sentence-transformers>=3.2.1',
    'transformers>=4.41,<5',
    'openai>=1.54.0',
    'requests>=2.32.3',
    'pandas>=2.2.3',
    'datasets>=3.1.0',
    'rank-bm25>=0.2.2',
    'tqdm>=4.66.5',
], check=True)
print('dependencies_ready')


In [ ]:
import json
import os
from dotenv import dotenv_values

def is_placeholder(value: str) -> bool:
    text = str(value or '').strip()
    return text in {'', 'your_jina_api_key'}

os.chdir(REPO_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
env_template_path = REPO_DIR / ENV_TEMPLATE_PATH
if not env_template_path.exists():
    raise FileNotFoundError(f'.env.example not found: {env_template_path}')

template_env = {
    str(key): '' if value is None else str(value)
    for key, value in dotenv_values(env_template_path).items()
}
for key, value in template_env.items():
    os.environ[str(key)] = str(value)

os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MILVUS_MODE'] = str(MILVUS_MODE).strip().lower()
if os.environ['MILVUS_MODE'] == 'lite':
    os.environ['MILVUS_LITE_URI'] = str(MILVUS_LITE_URI).strip()
    os.environ['MILVUS_URI'] = ''
    os.environ['MILVUS_TOKEN'] = ''
    os.environ['MILVUS_DB_NAME'] = ''
else:
    os.environ['MILVUS_URI'] = str(MILVUS_URI).strip()
    os.environ['MILVUS_TOKEN'] = str(MILVUS_TOKEN).strip()
    os.environ['MILVUS_DB_NAME'] = str(MILVUS_DB_NAME).strip()
    os.environ['MILVUS_LITE_URI'] = ''

embedding_key = str(EMBEDDING_API_KEY or JINA_API_KEY).strip()
reranker_key = str(RERANKER_API_KEY or JINA_API_KEY).strip()
if embedding_key:
    os.environ['EMBEDDING_API_KEY'] = embedding_key
if reranker_key:
    os.environ['RERANKER_API_KEY'] = reranker_key

for key, value in EXTRA_ENV_OVERRIDES.items():
    os.environ[str(key)] = str(value)

masked = {
    'env_template_path': str(env_template_path),
    'target': TARGET,
    'limit': LIMIT,
    'shuffle': SHUFFLE,
    'milvus_mode': os.environ.get('MILVUS_MODE', ''),
    'milvus_uri_set': bool(os.environ.get('MILVUS_URI')),
    'milvus_lite_uri': os.environ.get('MILVUS_LITE_URI', ''),
    'embedding_provider': os.environ.get('EMBEDDING_PROVIDER', ''),
    'embedding_model': os.environ.get('EMBEDDING_MODEL', ''),
    'embedding_api_key_set': not is_placeholder(os.environ.get('EMBEDDING_API_KEY', '')),
    'use_reranker': os.environ.get('USE_RERANKER', ''),
    'reranker_provider': os.environ.get('RERANKER_PROVIDER', ''),
    'reranker_model': os.environ.get('RERANKER_MODEL', ''),
    'reranker_api_key_set': not is_placeholder(os.environ.get('RERANKER_API_KEY', '')),
    'extra_env_overrides': EXTRA_ENV_OVERRIDES,
}
(OUTPUT_DIR / 'effective_public_eval_env.json').write_text(
    json.dumps(masked, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(masked, ensure_ascii=False, indent=2))

if os.environ['MILVUS_MODE'] == 'lite' and not os.environ.get('MILVUS_LITE_URI', '').strip():
    raise ValueError('MILVUS_LITE_URI is empty')
if os.environ['MILVUS_MODE'] != 'lite' and not os.environ.get('MILVUS_URI', '').strip():
    raise ValueError('MILVUS_URI is empty')
if os.environ.get('MILVUS_MODE', '') != 'lite' and not os.environ.get('MILVUS_TOKEN', '').strip():
    raise ValueError('MILVUS_TOKEN is empty')
if os.environ.get('EMBEDDING_PROVIDER', '').strip().lower() != 'huggingface' and is_placeholder(os.environ.get('EMBEDDING_API_KEY', '')):
    raise ValueError('EMBEDDING_API_KEY is empty or still using placeholder text')
if os.environ.get('USE_RERANKER', '').strip().lower() in {'1', 'true', 'yes', 'on'} and os.environ.get('RERANKER_PROVIDER', '').strip().lower() != 'huggingface' and is_placeholder(os.environ.get('RERANKER_API_KEY', '')):
    raise ValueError('RERANKER_API_KEY is empty or still using placeholder text')


In [ ]:
import json
import os
import time

def log_stage(message: str) -> None:
    now = time.strftime('%H:%M:%S')
    print(f'[{now}] {message}', flush=True)

start = time.perf_counter()
os.chdir(REPO_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
log_stage(f'Workspace ready: repo={REPO_DIR} output={OUTPUT_DIR}')
log_stage('Importing public paper benchmark modules...')
from config import load_config
from colab_eval.public_paper_benchmark import (
    run_public_paper_benchmark,
    save_public_paper_benchmark_result,
)

config = load_config()
active_config = {
    'embedding_provider': config.embedding_provider,
    'embedding_model': config.embedding_model,
    'vector_backend': config.vector_backend,
    'retrieval_mode': config.retrieval_mode,
    'retriever_top_k': config.retriever_top_k,
    'final_top_k': config.final_top_k,
    'query_rewrite_enabled': config.query_rewrite_enabled,
    'query_rewrite_max_variants': config.query_rewrite_max_variants,
    'use_reranker': config.use_reranker,
    'reranker_provider': config.reranker_provider,
    'reranker_model': config.reranker_model,
}
print(json.dumps(active_config, ensure_ascii=False, indent=2))

log_stage('Starting retrieval-only benchmark...')
result = run_public_paper_benchmark(
    config,
    target=TARGET,
    limit=LIMIT,
    seed=SEED,
    shuffle=SHUFFLE,
)
log_stage('Saving outputs...')
save_public_paper_benchmark_result(result, OUTPUT_DIR)
elapsed = time.perf_counter() - start

print('saved_to=', OUTPUT_DIR)
print('elapsed_sec=', round(elapsed, 2))
print(json.dumps(result.aggregate_summary, ensure_ascii=False, indent=2))


In [ ]:
import json
from pathlib import Path

summary_path = OUTPUT_DIR / 'public_paper_benchmark_summary.json'
print('summary_path=', summary_path)
print(summary_path.read_text(encoding='utf-8'))

print('\nGenerated files:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path)
